In [0]:
%skip
%sql
CREATE OR REFRESH MATERIALIZED VIEW oftalmo_sus.03_gold.mv_fato_atendimentos
COMMENT 'Dimensão geográfica dos municípios IBGE'
AS 
SELECT 
    *,
    CASE 
        --WHEN PA_IDADE IS NULL THEN NULL
        WHEN PA_IDADE < 18 THEN '0-17 anos'
        WHEN PA_IDADE BETWEEN 18 AND 59 THEN '18-59 anos'
        ELSE '60+ anos'
    END AS faixa_etaria
FROM oftalmo_sus.02_silver.tb_fato_sia_oftalmo;

In [0]:
from pyspark.sql.functions import col, when

# Lendo a Fato da Silver
df_silver_sia = spark.read.format("delta").load("oftalmo_sus.02_silver.tb_fato_sia_oftalmo")

# Aplicando a regra de negócio da Gold (Bucket de Faixa Etária)
df_gold_fato = df_silver_sia.withColumn("faixa_etaria",
    when(col("PA_IDADE") < 18, "0-17 anos")
    .when(col("PA_IDADE").between(18, 59), "18-59 anos")
    .otherwise("60+ anos")
)

# Gravando fisicamente na Gold
df_gold_fato.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("oftalmo_sus.03_gold.fato_atendimentos")

print("Tabela Fato de Atendimentos gravada com sucesso!")